# WordPiece Tokenizer Training for Quora Question Pairs

## Overview

This notebook trains a from-scratch **WordPiece** subword tokenizer on the [Quora Question Pairs](https://www.kaggle.com/c/quora-question-pairs) (QQP) dataset. QQP pairs two Quora questions with a binary `is_duplicate` label, and the underlying task — deciding whether two questions ask the same thing — is a form of paraphrase / semantic-equivalence detection. Before any duplicate-detection model can be trained, raw question text has to be converted into a fixed vocabulary of integer ids, and that conversion step is exactly what this notebook builds and validates.

The tokenizer produced here is a shared preprocessing component for a small family of duplicate-question models built on top of the same QQP corpus: an LSTM-Attention model, an ESIM-style model, and a custom Transformer encoder (the closing cell of the notebook states this explicitly). Because these architectures consume the tokenizer differently, the configuration carries **two separate sequence-length budgets** rather than one: a shorter budget for questions encoded independently (used by the LSTM-Attention and ESIM models), and a longer budget for both questions encoded together as a single `[CLS] question1 [SEP] question2 [SEP]` sequence (used by the Transformer).

## Workflow

The pipeline is organized as a linear sequence of stages:

1. **Configuration** — define every path and tokenizer hyperparameter as typed, frozen dataclasses.
2. **Data loading and cleaning** — load the raw QQP CSV and drop rows with missing or invalid values.
3. **Splitting and corpus building** — create a stratified train/validation split, then write a WordPiece training corpus from the training split only.
4. **Tokenizer training** — train a WordPiece model on the corpus and wrap it with BERT-style normalization, pre-tokenization, and post-processing.
5. **Wrapping and saving** — expose the trained backend through a Hugging Face `PreTrainedTokenizerFast` and persist every artifact to disk.
6. **Validation** — encode sample questions (both individually and as pairs) and check that ids, attention masks, and segment ids behave as expected.
7. **Save/reload consistency** — reload the tokenizer strictly from disk and confirm it reproduces identical encodings.
8. **Sequence-length analysis** — measure actual token-length distributions to check that the configured length budgets are reasonable.
9. **Metadata and final validation** — record every configuration choice and result to a JSON file, then re-verify the saved tokenizer end to end.

## Tools and data

- **[🤗 Tokenizers](https://github.com/huggingface/tokenizers)** builds and trains the WordPiece model itself (normalizer, pre-tokenizer, trainer, post-processor, decoder).
- **[🤗 Transformers](https://huggingface.co/docs/transformers)** wraps the trained backend as a `PreTrainedTokenizerFast` with a familiar padding/truncation API.
- **pandas** and **NumPy** handle the QQP DataFrame and the sequence-length statistics.
- **scikit-learn** provides the stratified train/validation split.
- The input is the QQP training CSV (`data/raw/train.csv`), expected to already be present on disk (this notebook does not download it).

## What this notebook produces

A tokenizer directory (`artifacts/tokenizers/wordpiece_uncased_30k/`) containing `tokenizer.json`, `tokenizer_config.json`, `vocab.txt`, `special_tokens_map.json`, and a `training_metadata.json` file documenting exactly how the tokenizer was trained. Train/validation CSV splits and the raw training corpus are also written to `data/processed/` so that downstream model-training notebooks can reuse the identical split.

## Section 1 - Environment Setup and Configuration

This section defines every path and hyperparameter the rest of the notebook depends on, before any data is touched. Keeping configuration in two small, frozen dataclasses — one for file paths, one for tokenizer hyperparameters — means every later cell reads from `paths.*` or `cfg.*` instead of repeating string literals or magic numbers.

### 1.1 Library Imports

The imports cover four areas: `dataclasses` for the typed configuration objects, `pandas`/`numpy`/`scikit-learn` for loading and splitting the QQP data, and the `tokenizers` + `transformers` building blocks (`Tokenizer`, `WordPiece`, `BertNormalizer`, `BertPreTokenizer`, `TemplateProcessing`, `WordPieceTrainer`, `PreTrainedTokenizerFast`) used to construct, train, and wrap the tokenizer itself.

In [1]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version as package_version
import json
import os
from pathlib import Path
import platform
import random
from typing import Any

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from tokenizers import Tokenizer
from tokenizers.decoders import WordPiece as WordPieceDecoder
from tokenizers.models import WordPiece
from tokenizers.normalizers import BertNormalizer
from tokenizers.pre_tokenizers import BertPreTokenizer
from tokenizers.processors import TemplateProcessing
from tokenizers.trainers import WordPieceTrainer

from transformers import PreTrainedTokenizerFast


### 1.2 Path Configuration

`WordPiecePaths` is a frozen dataclass that centralizes every file and directory the pipeline touches — the raw input CSV, the processed train/validation splits and training corpus, and the final tokenizer artifact directory — behind read-only `@property` accessors. Two small guard methods, `make_dirs` and `validate_input`, are run immediately after instantiation (in the next cell) so that missing output directories or a missing input CSV are caught before any real work starts.

In [2]:
@dataclass(frozen=True)
class WordPiecePaths:
    """Paths used by the QQP WordPiece-tokenizer pipeline."""

    train_csv_path: Path = Path("data/raw/train.csv")
    processed_dir: Path = Path("data/processed")
    tokenizer_dir: Path = Path(
        "artifacts/tokenizers/wordpiece_uncased_30k"
    )

    @property
    def corpus_path(self) -> Path:
        """Path to the training corpus text file consumed by the WordPiece trainer."""
        return self.processed_dir / "tokenizer_corpus.txt"

    @property
    def train_split_path(self) -> Path:
        """Path to the saved training-split CSV."""
        return self.processed_dir / "train_split.csv"

    @property
    def valid_split_path(self) -> Path:
        """Path to the saved validation-split CSV."""
        return self.processed_dir / "valid_split.csv"

    @property
    def tokenizer_json_path(self) -> Path:
        """Path to the saved ``tokenizer.json`` file."""
        return self.tokenizer_dir / "tokenizer.json"

    @property
    def tokenizer_config_path(self) -> Path:
        """Path to the saved ``tokenizer_config.json`` file."""
        return self.tokenizer_dir / "tokenizer_config.json"

    @property
    def special_tokens_map_path(self) -> Path:
        """Path to the saved ``special_tokens_map.json`` file."""
        return self.tokenizer_dir / "special_tokens_map.json"

    @property
    def vocab_path(self) -> Path:
        """Path to the saved WordPiece vocabulary file (``vocab.txt``)."""
        return self.tokenizer_dir / "vocab.txt"

    @property
    def training_metadata_path(self) -> Path:
        """Path to the saved tokenizer training metadata JSON file."""
        return self.tokenizer_dir / "training_metadata.json"

    def make_dirs(self) -> None:
        """Create all output directories."""
        self.processed_dir.mkdir(parents=True, exist_ok=True)
        self.tokenizer_dir.mkdir(parents=True, exist_ok=True)
        print(">>> Directories created...")

    def validate_input(self) -> None:
        """Check that the original QQP training CSV exists."""
        if not self.train_csv_path.is_file():
            raise FileNotFoundError(
                f">>> QQP training CSV was not found: {self.train_csv_path}"
            )


### 1.3 Tokenizer Configuration

`WordPieceConfig` is the single source of truth for every tokenizer hyperparameter: the target vocabulary size (30,000) and minimum merge frequency used by the WordPiece trainer, BERT-style normalization flags (lowercasing, accent stripping, Chinese-character handling), the special-token strings, and the train/validation split fraction and random seed.

Two fields deserve a closer look, since they drive a decision that shows up repeatedly later in the notebook: `max_sequence_length` (64) and `max_pair_length` (128) are two independent length budgets for the two ways this tokenizer's output will be consumed — a single question encoded on its own (`max_sequence_length`, for the LSTM-Attention and ESIM models) versus two questions encoded together as one `[CLS] q1 [SEP] q2 [SEP]` sequence (`max_pair_length`, for the custom Transformer). Keeping both on the same config object means every later cell can reference `cfg.max_sequence_length` or `cfg.max_pair_length` directly instead of hard-coding a number.

In [3]:
@dataclass(frozen=True)
class WordPieceConfig:
    """Configuration for training the QQP WordPiece tokenizer."""

    # WordPiece vocabulary
    vocab_size: int = 30_000
    min_frequency: int = 2
    continuing_subword_prefix: str = "##"
    max_input_chars_per_word: int = 100

    # Normalization
    lowercase: bool = True
    strip_accents: bool = True
    clean_text: bool = True
    handle_chinese_chars: bool = True

    # LSTM-Attention and ESIM:
    # each question is encoded separately
    max_sequence_length: int = 64

    # Custom Transformer:
    # both questions are encoded together
    max_pair_length: int = 128

    # Special tokens
    pad_token: str = "[PAD]"
    unk_token: str = "[UNK]"
    cls_token: str = "[CLS]"
    sep_token: str = "[SEP]"
    mask_token: str = "[MASK]"

    # Data split
    valid_size: float = 0.10
    seed: int = 28

    # Used during length analysis
    length_analysis_batch_size: int = 4096

    @property
    def special_tokens(self) -> list[str]:
        """Return special tokens in deterministic order."""
        return [
            self.pad_token,
            self.unk_token,
            self.cls_token,
            self.sep_token,
            self.mask_token,
        ]

### 1.4 Instantiating and Validating

`cfg` and `paths` are instantiated with their defaults, the output directories are created, and the presence of the raw QQP CSV is checked — a deliberate fail-fast step so a missing input file raises immediately rather than partway through data loading.

In [4]:
cfg = WordPieceConfig()
paths = WordPiecePaths()

paths.make_dirs()
paths.validate_input()

>>> Directories created...


## Section 2 - Reproducibility and Utility Functions

Three small helpers are defined here and reused throughout the rest of the notebook. `seed_everything` seeds Python's and NumPy's random number generators and fixes `PYTHONHASHSEED`; note that the stratified split in Section 4 uses scikit-learn's own `random_state` argument rather than this global seed, so `seed_everything` mainly protects any other NumPy-based randomness in the pipeline. `save_json` and `load_json` are thin, reusable wrappers around `json.dump`/`json.load` that also create missing parent directories — they are used again in Section 11 to persist the tokenizer training metadata.

> **Note:** `save_json` and `load_json` are annotated with `Dict[str, Any]`, but only `Any` is imported from `typing` in Section 1 (`Dict` and, later, `Tuple` are used in annotations without being imported). This does not raise an error here because `from __future__ import annotations` turns every annotation into a string that Python never evaluates at runtime — it would only surface if something later called `typing.get_type_hints()` on these functions.

In [5]:
def seed_everything(seed: int) -> None:
    """Seed Python, NumPy, and the process hash seed for reproducibility.

    Args:
        seed: Seed value applied to ``PYTHONHASHSEED`` and to NumPy's and
            Python's random number generators.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    random.seed(seed)

    print(f">>> Seed set to {seed}...")

def save_json(data: Dict[str, Any], path: str | Path) -> None:
    """Write a dictionary to disk as a pretty-printed JSON file.

    Creates any missing parent directories before writing.

    Args:
        data: Mapping to serialize to JSON.
        path: Destination file path; parent directories are created if
            they do not already exist.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f">>> JSON saved to {path}...")

def load_json(path: str | Path) -> Dict[str, Any]:
    """Load and parse a JSON file from disk.

    Args:
        path: Path to the JSON file to read.

    Returns:
        The parsed JSON content as a dictionary.
    """
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"JSON file loaded from {path}...")
    return data


seed_everything(cfg.seed)

>>> Seed set to 28...


## Section 3 - Loading and Cleaning the QQP Dataset

The Quora Question Pairs dataset contains roughly 404K English question pairs, each labeled `1` if the two questions are duplicates (semantically equivalent) and `0` otherwise. `load_qqp_data` keeps only the three required columns (`question1`, `question2`, `is_duplicate`), coerces the label to numeric so malformed values become `NaN`, drops any row missing a question or a label, collapses repeated whitespace in the question text, and casts the label to `int8`.

The two cells that follow run this loader and inspect the result: a `.head()` preview of the cleaned DataFrame, and a class-balance / schema check (`value_counts` plus `df.info()`) confirming the roughly 63% non-duplicate / 37% duplicate split and the expected column dtypes before any further processing happens.

In [6]:
def load_qqp_data(
    csv_path: str | Path,
) -> pd.DataFrame:
    """Load the QQP training CSV and remove invalid rows.

    Keeps only ``question1``, ``question2``, and ``is_duplicate``, coerces
    ``is_duplicate`` to numeric (turning unparseable labels into NaN), drops
    rows with a missing question or label, collapses repeated whitespace in
    the question text, and keeps only rows whose label is 0 or 1.

    Args:
        csv_path: Path to the QQP training CSV file.

    Returns:
        A cleaned DataFrame with columns ``question1``, ``question2``, and
        ``is_duplicate`` (as ``int8``), indexed from 0.
    """

    csv_path = Path(csv_path)

    df = pd.read_csv(csv_path)

    required_columns = [
        "question1",
        "question2",
        "is_duplicate",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required QQP columns: {missing_columns}"
        )

    original_rows = len(df)

    # Keep only the required columns.
    df = df[required_columns].copy()

    # Invalid labels become NaN.
    df["is_duplicate"] = pd.to_numeric(
        df["is_duplicate"],
        errors="coerce",
    )

    # Both questions and the label are required.
    df = df.dropna(
        subset=[
            "question1",
            "question2",
            "is_duplicate",
        ]
    ).copy()

    # Remove surrounding and repeated whitespace.
    for column in ["question1", "question2"]:
        df[column] = (
            df[column]
            .astype(str)
            .str.replace(
                r"\s+",
                " ",
                regex=True,
            )
            .str.strip()
        )

    valid_questions = (
        df["question1"].ne("")
        & df["question2"].ne("")
    )

    valid_labels = df["is_duplicate"].isin(
        [0, 1]
    )

    df = df.loc[
        valid_questions & valid_labels,
        required_columns,
    ].copy()

    df["is_duplicate"] = (
        df["is_duplicate"].astype(np.int8)
    )

    df = df.reset_index(drop=True)

    removed_rows = original_rows - len(df)

    print(
        f">>> QQP loaded from: {csv_path}\n"
        f">>> Original rows: {original_rows:,}\n"
        f">>> Valid rows: {len(df):,}\n"
        f">>> Removed invalid rows: {removed_rows:,}"
    )

    return df

In [7]:
df = load_qqp_data(paths.train_csv_path)
df.head()

>>> QQP loaded from: data\raw\train.csv
>>> Original rows: 404,290
>>> Valid rows: 404,287
>>> Removed invalid rows: 3


,question1,question2,is_duplicate
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [8]:
print(df["is_duplicate"].value_counts(
    normalize=True
), "\n")
df.info()

is_duplicate
0    0.630799
1    0.369201
Name: proportion, dtype: float64 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404287 entries, 0 to 404286
Data columns (total 3 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   question1     404287 non-null  object
 1   question2     404287 non-null  object
 2   is_duplicate  404287 non-null  int8  
dtypes: int8(1), object(2)
memory usage: 6.6+ MB


## Section 4 - Building the Train/Validation Split and Tokenizer Corpus

`create_train_valid_split` uses scikit-learn's `train_test_split` with `stratify=df["is_duplicate"]` to carve out a 90/10 train/validation split (`cfg.valid_size`) that preserves the original class balance in both halves, then resets each DataFrame's index. `save_splits` writes both splits to CSV in `data/processed/` so that downstream model-training notebooks can reuse the exact same split rather than re-sampling it.

`build_corpus_file` then writes the WordPiece training corpus — but only from `train_df`, not the full dataset. This is a deliberate choice: the validation split is held out from vocabulary learning entirely, the same way it should be held out from any other component fit during training, so that no information about the validation set leaks into the tokenizer itself. Each row contributes `question1` and `question2` as two separate lines, so the trainer sees roughly 728K individual question lines even though only about 364K rows are used.

> **Note:** the final `print` statement inside `build_corpus_file` concatenates two f-strings without a separating newline, so in the output of the next cell the corpus path and the question count are printed running together on one line (`...tokenizer_corpus.txt>>> Question counts: 727,716`). This is purely a cosmetic formatting issue in the printed message — the corpus file itself and its contents are unaffected.

In [9]:
def create_train_valid_split(
    df: pd.DataFrame,
    cfg: WordPieceConfig
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    """Split the cleaned QQP DataFrame into stratified train/validation sets.

    Uses scikit-learn's ``train_test_split`` with stratification on
    ``is_duplicate`` so both splits preserve the original class balance,
    and resets the row index on each resulting DataFrame.

    Args:
        df: Cleaned QQP DataFrame containing an ``is_duplicate`` column.
        cfg: Tokenizer configuration; supplies the validation fraction
            (``valid_size``) and the random seed (``seed``) used for the
            split.

    Returns:
        A tuple of ``(train_df, val_df)`` with reset indices.
    """
    train_df, val_df = train_test_split(
        df,
        test_size=cfg.valid_size,
        random_state=cfg.seed,
        stratify=df["is_duplicate"]
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    print(
        f">>> DataFrame splitted to train and validation...\n"
        f">>> Training rows: {len(train_df):,}\n"
        f">>> Validation rows: {len(val_df):,}\n"
        f">>> Train duplicate rate: "
        f"{train_df['is_duplicate'].mean():.4f}\n"
        f">>> Validation duplicate rate: "
        f"{val_df['is_duplicate'].mean():.4f}"
    )

    return train_df, val_df

In [10]:
def save_splits(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    paths: WordPiecePaths
) -> None:
    """Persist the train and validation splits to CSV.

    Args:
        train_df: Training split to save.
        val_df: Validation split to save.
        paths: Path configuration; supplies ``train_split_path`` and
            ``valid_split_path`` as the CSV destinations.
    """
    train_df.to_csv(paths.train_split_path, index=False)
    val_df.to_csv(paths.valid_split_path, index=False)

    print(
        f">>> Train DataFrame saved to: {paths.train_split_path}\n"
        f">>> Validation DataFrame saved to: {paths.valid_split_path}"
    )

In [47]:
def build_corpus_file(
    train_df: pd.DataFrame,
    paths: WordPiecePaths
) -> int:

    """Write the WordPiece training corpus from the training split.

    Each row contributes both ``question1`` and ``question2`` as separate
    lines, so the tokenizer trainer sees every question independently
    rather than as a pair.

    Args:
        train_df: Training split providing the ``question1`` and
            ``question2`` columns to write.
        paths: Path configuration; supplies ``corpus_path`` as the output
            file location.

    Returns:
        The total number of question lines written to the corpus file.
    """
    question_count = 0
    with open(paths.corpus_path, "w", encoding="utf-8") as f:
        for row in train_df.itertuples(index=False):
            f.write(f"{row.question1}\n")
            f.write(f"{row.question2}\n")
            question_count += 2

    print(
        f">>> Tokenizer corpus saved to: {paths.corpus_path}"
        f">>> Question counts: {question_count:,}"
    )

    return question_count

In [48]:
train_df, val_df = create_train_valid_split(
    df=df, cfg=cfg
)
save_splits(
    train_df=train_df,
    val_df=val_df,
    paths=paths
)
corpus_question_count = build_corpus_file(
    train_df=train_df,
    paths=paths,
)

>>> DataFrame splitted to train and validation...
>>> Training rows: 363,858
>>> Validation rows: 40,429
>>> Train duplicate rate: 0.3692
>>> Validation duplicate rate: 0.3692
>>> Train DataFrame saved to: data\processed\train_split.csv
>>> Validation DataFrame saved to: data\processed\valid_split.csv
>>> Tokenizer corpus saved to: data\processed\tokenizer_corpus.txt>>> Question counts: 727,716


In [13]:
with paths.corpus_path.open(
    "r",
    encoding="utf-8",
) as file:
    for _ in range(5):
        print(file.readline().strip())

What is maximum steering wheel torque in truck?
What is a steering wheel's torque?
How much does it cost to build a website in India?
How much does it cost to build an ecommerce store in India?
Does time stop ever?


## Section 5 - Training the WordPiece Backend Tokenizer

`train_wordpiece_backend` is where the actual subword vocabulary is learned. It assembles a Hugging Face `tokenizers.Tokenizer` pipeline piece by piece:

1. A `WordPiece` model, configured with the unknown-token string, the `##` continuing-subword prefix, and a maximum character length per word.
2. A `BertNormalizer` that applies the cleaning, accent-stripping, Chinese-character handling, and lowercasing flags from `cfg`.
3. A `BertPreTokenizer`, which splits text on whitespace and punctuation before any subword learning happens.
4. A `WordPieceTrainer`, configured with the target vocabulary size, minimum merge frequency, and the special tokens that must always receive their own vocabulary entries.
5. `tokenizer.train(...)`, which runs the actual training pass over the corpus file built in Section 4.

Unlike BPE, which always merges the *most frequent* adjacent pair, WordPiece scores each candidate pair and merges the one with the highest score:

$$
\text{score}(a, b) = \frac{\text{freq}(a, b)}{\text{freq}(a) \times \text{freq}(b)}
$$

Dividing the pair's frequency by the product of its parts' individual frequencies means a pair is preferred when its two pieces are individually *rare* — this keeps common short subwords (like frequent word fragments) from being merged away too eagerly, even if they co-occur often ([Hugging Face LLM Course, "WordPiece tokenization"](https://huggingface.co/learn/llm-course/chapter6/6)).

After training, the function checks that every configured special token actually received a vocabulary id (raising `RuntimeError` if not — a defensive check against silent misconfiguration), then attaches a `TemplateProcessing` post-processor that wraps a single question as `[CLS] $A [SEP]` and a pair of questions as `[CLS] $A [SEP] $B:1 [SEP]:1`. The `:1` suffix on the second question and its `[SEP]` is what produces the `token_type_ids` (segment ids) used later to distinguish "question A" tokens from "question B" tokens. Finally, a `WordPieceDecoder` is attached so that `##`-prefixed subwords can be reassembled back into words when decoding.

In [14]:
def train_wordpiece_backend(
    paths: WordPiecePaths,
    cfg: WordPieceConfig
) -> Tokenizer:
    """Train a WordPiece tokenizer backend on the corpus file.

    Builds a Hugging Face ``tokenizers.Tokenizer`` around a ``WordPiece``
    model with BERT-style normalization (lowercasing, accent stripping)
    and pre-tokenization, trains it on ``paths.corpus_path`` with a
    ``WordPieceTrainer``, verifies that every configured special token was
    assigned an id, and attaches a ``TemplateProcessing`` post-processor so
    single and paired inputs are wrapped with ``[CLS]``/``[SEP]`` tokens
    (with paired inputs carrying distinct segment ids).

    Args:
        paths: Path configuration; supplies ``corpus_path`` as the training
            corpus.
        cfg: Tokenizer configuration; supplies the vocabulary size,
            minimum merge frequency, normalization flags, and special
            tokens used to configure the model and trainer.

    Returns:
        The trained ``Tokenizer`` backend, ready to be wrapped in a
        ``PreTrainedTokenizerFast``.

    Raises:
        RuntimeError: If any configured special token is missing from the
            trained vocabulary.
    """
    tokenizer = Tokenizer(
        WordPiece(
            unk_token=cfg.unk_token,
            continuing_subword_prefix=(
                cfg.continuing_subword_prefix
            ),
            max_input_chars_per_word=(
                cfg.max_input_chars_per_word
            ),
        )
    )

    tokenizer.normalizer = BertNormalizer(
        clean_text=cfg.clean_text,
        handle_chinese_chars=(
            cfg.handle_chinese_chars
        ),
        strip_accents=cfg.strip_accents,
        lowercase=cfg.lowercase
    )

    tokenizer.pre_tokenizer = (
        BertPreTokenizer()
    )

    trainer = WordPieceTrainer(
        vocab_size=cfg.vocab_size,
        min_frequency=cfg.min_frequency,
        show_progress=True,
        special_tokens=cfg.special_tokens,
        continuing_subword_prefix=(
            cfg.continuing_subword_prefix
        )
    )

    tokenizer.train(
        files=[
            str(paths.corpus_path)
        ],
        trainer=trainer
    )

    special_token_ids = {
        token: tokenizer.token_to_id(token)
        for token in cfg.special_tokens
    }

    missing_special_tokens = [
        token for token, token_id in special_token_ids.items()
        if token_id is None
    ]

    if missing_special_tokens:
        raise RuntimeError(
            "The trained vocabulary is missing "
            f"special tokens: {missing_special_tokens}"
        )

    tokenizer.post_processor = (
        TemplateProcessing(
            single=(
                f"{cfg.cls_token} "
                f"$A "
                f"{cfg.sep_token}"
            ),
            pair=(
                f"{cfg.cls_token} "
                f"$A "
                f"{cfg.sep_token} "
                f"$B:1 "
                f"{cfg.sep_token}:1"
            ),
            special_tokens=[
                (
                    cfg.cls_token,
                    special_token_ids[
                        cfg.cls_token
                    ],
                ),
                (
                    cfg.sep_token,
                    special_token_ids[
                        cfg.sep_token
                    ],
                ),
            ],
        )
    )

    tokenizer.decoder = WordPieceDecoder(
        prefix=cfg.continuing_subword_prefix
    )

    print(
        ">>> WordPiece backend trained.\n"
        f">>> Actual vocabulary size: "
        f"{tokenizer.get_vocab_size():,}"
    )

    return tokenizer

In [15]:
backend_tokenizer = train_wordpiece_backend(
    paths=paths,
    cfg=cfg,
)

>>> WordPiece backend trained.
>>> Actual vocabulary size: 30,000


### 5.1 Quick Encoding Checks

Two lightweight smoke tests confirm the freshly trained backend behaves as configured, before any wrapping or saving happens. The single-sequence example shows the WordPiece split in action — "Tokenization" becomes `token` + `##ization` — while the paired example shows `type_ids` switching from `0` to `1` at the second question, exactly as configured by the pair template in Section 5.

In [16]:
sample_encoding = backend_tokenizer.encode(
    "Tokenization is useful for unseen words."
)
print(sample_encoding.tokens)
print(sample_encoding.ids)

['[CLS]', 'token', '##ization', 'is', 'useful', 'for', 'unseen', 'words', '.', '[SEP]']
[2, 21983, 2610, 1460, 3832, 1484, 27775, 3379, 18, 3]


In [17]:
pair_encoding = backend_tokenizer.encode(
    "How can I learn machine learning?",
    "What is the best way to study machine learning?",
)
print(pair_encoding.tokens)
print(pair_encoding.type_ids)

['[CLS]', 'how', 'can', 'i', 'learn', 'machine', 'learning', '?', '[SEP]', 'what', 'is', 'the', 'best', 'way', 'to', 'study', 'machine', 'learning', '?', '[SEP]']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## Section 6 - Wrapping as a Hugging Face Fast Tokenizer

`create_fast_tokenizer` wraps the raw `tokenizers.Tokenizer` backend in a `transformers.PreTrainedTokenizerFast`, which exposes the familiar callable API (`tokenizer(text, padding=..., truncation=...)`) and the `save_pretrained` / `from_pretrained` methods used throughout the Transformers ecosystem. `model_max_length` is set from `cfg.max_pair_length` — the larger of the two configured budgets, since paired encoding is the longer of the two use cases — and both padding and truncation are pinned to the right side. A final check confirms `tokenizer.is_fast` is `True`, guarding against the Rust-backed implementation silently failing to attach. The `special_tokens_map` printed afterward is just a quick visual confirmation that all five special tokens (`[PAD]`, `[UNK]`, `[CLS]`, `[SEP]`, `[MASK]`) are correctly registered on the wrapper.

In [18]:
def create_fast_tokenizer(
    backend_tokenizer: Tokenizer,
    cfg: WordPieceConfig
) -> PreTrainedTokenizerFast:

    """Wrap a trained WordPiece backend in a Hugging Face fast tokenizer.

    Args:
        backend_tokenizer: Trained ``tokenizers.Tokenizer`` backend to wrap.
        cfg: Tokenizer configuration; supplies the special token strings
            and ``max_pair_length``, used as the tokenizer's
            ``model_max_length``.

    Returns:
        A ``PreTrainedTokenizerFast`` configured with right-side padding
        and truncation.

    Raises:
        RuntimeError: If the resulting tokenizer is not backed by the
            Rust fast-tokenizer implementation.
    """
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=backend_tokenizer,
        unk_token=cfg.unk_token,
        cls_token=cfg.cls_token,
        sep_token=cfg.sep_token,
        pad_token=cfg.pad_token,
        mask_token=cfg.mask_token,
        model_max_length=cfg.max_pair_length
    )

    tokenizer.padding_side = "right"
    tokenizer.truncation_side = "right"

    if not tokenizer.is_fast:
        raise RuntimeError(
            ">>> Expected a Fast Tokenizer."
        )

    print(
        ">>> Fast tokenizer created.\n"
        f">>> Class: {type(tokenizer).__name__}\n"
        f">>> Vocabulary size: {len(tokenizer):,}"
    )

    return tokenizer

In [19]:
tokenizer = create_fast_tokenizer(
    backend_tokenizer=backend_tokenizer,
    cfg=cfg
)

>>> Fast tokenizer created.
>>> Class: PreTrainedTokenizerFast
>>> Vocabulary size: 30,000


In [20]:
tokenizer.special_tokens_map

{'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]'}

## Section 7 - Saving the Tokenizer Artifacts

`save_tokenizer` saves the tokenizer twice, covering two different consumption paths: `tokenizer.save_pretrained(...)` writes the files a `transformers`-based training script expects (`tokenizer.json`, `tokenizer_config.json`, and, depending on the installed library version, `special_tokens_map.json`), while `backend_tokenizer.model.save(...)` writes the raw WordPiece vocabulary as `vocab.txt`. The function then checks that the required files actually landed on disk and prints everything that was saved, so a partial or failed save is caught immediately rather than discovered later by a downstream notebook.

In [25]:
def save_tokenizer(
    backend_tokenizer: Tokenizer,
    tokenizer: PreTrainedTokenizerFast,
    paths: WordPiecePaths
) -> list[str]:

    """Save the tokenizer to disk and verify the expected files exist.

    Saves both the ``PreTrainedTokenizerFast`` (via ``save_pretrained``)
    and the underlying WordPiece model vocabulary to
    ``paths.tokenizer_dir``, then checks that the tokenizer JSON, config,
    and vocabulary files were written.

    Args:
        backend_tokenizer: Trained ``tokenizers.Tokenizer`` backend whose
            model vocabulary is saved alongside the fast tokenizer.
        tokenizer: Fast tokenizer wrapper to save.
        paths: Path configuration; supplies ``tokenizer_dir`` as the
            output directory and the expected output file paths.

    Returns:
        Sorted list of file names actually present in ``tokenizer_dir``
        after saving.

    Raises:
        FileNotFoundError: If any of the required tokenizer files
            (``tokenizer.json``, ``tokenizer_config.json``,
            ``vocab.txt``) is missing after saving.
    """
    paths.tokenizer_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    tokenizer.save_pretrained(
        str(paths.tokenizer_dir)
    )

    backend_tokenizer.model.save(
        str(paths.tokenizer_dir)
    )

    required_paths = [
        paths.tokenizer_json_path,
        paths.tokenizer_config_path,
        paths.vocab_path
    ]

    missing_files = [
        path.name for path in required_paths if not path.is_file()
    ]

    if missing_files:
        raise FileNotFoundError(
            f">>> Missing tokenizer files: {missing_files}"
        )

    saved_files = sorted(
        path.name for path in paths.tokenizer_dir.iterdir() if path.is_file()
    )

    print(">>> Saved tokenizer files:")

    for filename in saved_files:
        print(f"    - {filename}")

    if not paths.special_tokens_map_path.exists():
        print(
            ">>> special_tokens_map.json was not "
            "created by this Transformers version. "
            "This is acceptable if reload validation passes."
        )

    return saved_files

In [26]:
saved_files = save_tokenizer(
    backend_tokenizer=backend_tokenizer,
    tokenizer=tokenizer,
    paths=paths,
)

>>> Saved tokenizer files:
    - special_tokens_map.json
    - tokenizer.json
    - tokenizer_config.json
    - vocab.txt


## Section 8 - Validating Tokenizer Behavior

This section runs a block of inline checks — effectively lightweight unit tests — confirming the saved tokenizer produces exactly the input shapes the downstream models expect: fixed-length, padded and truncated `input_ids` and `attention_mask` arrays for a single question, and `[CLS]`/`[SEP]`-delimited sequences with correct segment ids for a pair of questions. The first example deliberately includes a URL fragment (`www.chatgpt.com`) to see how the trained vocabulary handles text it is unlikely to have seen much of during training — it gets broken down into smaller WordPiece fragments (`chat`, `##g`, `##pt`) rather than mapped to `[UNK]`.

In [28]:
question = "How can I learn machine learning on www.chatgpt.com"

separated_encoding = tokenizer(
    question,
    add_special_tokens=False,
    padding="max_length",
    truncation=True,
    max_length=cfg.max_sequence_length,
    return_attention_mask=True
)

print(
    tokenizer.convert_ids_to_tokens(
        separated_encoding["input_ids"]
    )
)

print(separated_encoding["attention_mask"])

['how', 'can', 'i', 'learn', 'machine', 'learning', 'on', 'www', '.', 'chat', '##g', '##pt', '.', 'com', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [30]:
assert (
    len(separated_encoding["input_ids"])
    == cfg.max_sequence_length
)

assert (
    len(separated_encoding["attention_mask"])
    == cfg.max_sequence_length
)

print(">>> Separate-question encoding passed.")

>>> Separate-question encoding passed.


### 8.1 Paired-Input Encoding and Segment IDs

`token_type_ids` (segment ids) are `0` for the first question and its closing `[SEP]`, and `1` for the second question and its `[SEP]` — this is exactly the pair template configured back in Section 5, and it's what lets a Transformer encoder tell "question A" tokens apart from "question B" tokens inside one concatenated sequence.

In [32]:
question1 = "How can I learn machine learning?"

question2 = (
    "What is the best way to study machine learning?"
)

pair_encoding = tokenizer(
    question1,
    question2,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=cfg.max_pair_length,
    return_attention_mask=True,
    return_token_type_ids=True
)

active_length = int(sum(pair_encoding["attention_mask"]))
active_ids = pair_encoding["input_ids"][:active_length]
active_tokens = tokenizer.convert_ids_to_tokens(active_ids)
active_type_ids = pair_encoding["token_type_ids"][:active_length]
print(active_tokens, "-->")
print(active_ids)
print(f"Token Type IDs: {active_type_ids}")

['[CLS]', 'how', 'can', 'i', 'learn', 'machine', 'learning', '?', '[SEP]', 'what', 'is', 'the', 'best', 'way', 'to', 'study', 'machine', 'learning', '?', '[SEP]'] -->
[2, 1467, 1481, 51, 1681, 3010, 2202, 35, 3, 1455, 1460, 1449, 1508, 1627, 1464, 2063, 3010, 2202, 35, 3]
Token Type IDs: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [33]:
assert active_tokens[0] == cfg.cls_token

assert (
    active_tokens.count(cfg.sep_token)
    == 2
)

first_sep_index = active_tokens.index(
    cfg.sep_token
)

assert all(
    token_type_id == 0
    for token_type_id in active_type_ids[
        : first_sep_index + 1
    ]
)

assert all(
    token_type_id == 1
    for token_type_id in active_type_ids[
        first_sep_index + 1 :
    ]
)

assert (
    len(pair_encoding["input_ids"])
    == cfg.max_pair_length
)

assert (
    len(pair_encoding["attention_mask"])
    == cfg.max_pair_length
)

assert (
    len(pair_encoding["token_type_ids"])
    == cfg.max_pair_length
)

print(">>> Paired encoding validation passed.")

>>> Paired encoding validation passed.


### 8.2 Special-Token ID Consistency

This check confirms every special token resolved to a distinct, non-`None` id after loading the tokenizer through the Transformers wrapper — guarding against a silent misconfiguration where two special tokens would otherwise collide on the same id.

In [34]:
special_token_ids = {
    "pad": tokenizer.pad_token_id,
    "unk": tokenizer.unk_token_id,
    "cls": tokenizer.cls_token_id,
    "sep": tokenizer.sep_token_id,
    "mask": tokenizer.mask_token_id,
}
print(special_token_ids)

{'pad': 0, 'unk': 1, 'cls': 2, 'sep': 3, 'mask': 4}


In [35]:
assert all(
    token_id is not None
    for token_id in special_token_ids.values()
)

assert (
    len(set(special_token_ids.values()))
    == len(special_token_ids)
)

print(">>> Special-token validation passed.")

>>> Special-token validation passed.


## Section 9 - Save/Reload Consistency Check

The tokenizer is reloaded strictly from the files just written to disk (`local_files_only=True`, so no network access is involved), and a fresh pair of test questions is encoded with both the in-memory tokenizer and the reloaded one. Every field — `input_ids`, `attention_mask`, `token_type_ids`, and `special_tokens_map` — is asserted to be identical between the two. This is the round-trip check that matters in practice: it proves that whatever gets committed to disk and later loaded by a separate model-training script behaves exactly the same as the tokenizer trained a moment ago in this session.

In [36]:
reloaded_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    str(paths.tokenizer_dir),
    local_files_only=True
)

In [37]:
test_q1 = "Why is the sky blue?"

test_q2 = "What causes the sky to appear blue?"

encoding_kwargs = {
    "add_special_tokens": True,
    "padding": "max_length",
    "truncation": True,
    "max_length": cfg.max_pair_length,
    "return_attention_mask": True,
    "return_token_type_ids": True,
}

original_encoding = tokenizer(
    test_q1,
    test_q2,
    **encoding_kwargs,
)

reloaded_encoding = reloaded_tokenizer(
    test_q1,
    test_q2,
    **encoding_kwargs,
)

assert (
    original_encoding["input_ids"]
    == reloaded_encoding["input_ids"]
)

assert (
    original_encoding["attention_mask"]
    == reloaded_encoding["attention_mask"]
)

assert (
    original_encoding["token_type_ids"]
    == reloaded_encoding["token_type_ids"]
)

assert (
    tokenizer.special_tokens_map
    == reloaded_tokenizer.special_tokens_map
)

print(">>> Save and reload validation passed.")

>>> Save and reload validation passed.


## Section 10 - Sequence Length Analysis

Having trained, saved, and validated the tokenizer, this section goes back and checks empirically whether the two length budgets chosen in Section 1 — `max_sequence_length=64` and `max_pair_length=128` — were reasonable choices, rather than assuming they were.

`summarize_lengths` computes count, mean, the 50th/90th/95th/99th percentiles, the maximum, and the count and percentage of examples exceeding a given limit. `analyze_sequence_lengths` applies this to four groups: `question1` alone, `question2` alone, the two combined (`all_separate_questions`), and the full `[CLS] q1 [SEP] q2 [SEP]` pairs (`paired_inputs`) — encoding the questions in batches (`cfg.length_analysis_batch_size`) rather than one row at a time, purely for speed. The resulting table is the evidence used to judge whether the configured limits leave enough headroom.

In [39]:
def summarize_lengths(
    lengths: list[int],
    configured_limit: int,
) -> dict[str, int | float]:
    """Calculate summary statistics for a list of token lengths.

    Args:
        lengths: Token counts to summarize, one value per example.
        configured_limit: Sequence-length limit to compare the lengths
            against, used to report how many examples would be
            truncated.

    Returns:
        A dictionary with the sample count, mean, the 50th/90th/95th/99th
        percentiles (ceiling-rounded), the maximum length, the configured
        limit, and the count and percentage of examples exceeding it.
    """

    values = np.asarray(
        lengths,
        dtype=np.int32,
    )

    percentiles = np.percentile(
        values,
        [50, 90, 95, 99],
    )

    over_limit = int(
        np.count_nonzero(
            values > configured_limit
        )
    )

    return {
        "count": int(values.size),
        "mean": round(
            float(values.mean()),
            3,
        ),
        "p50": int(np.ceil(percentiles[0])),
        "p90": int(np.ceil(percentiles[1])),
        "p95": int(np.ceil(percentiles[2])),
        "p99": int(np.ceil(percentiles[3])),
        "max": int(values.max()),
        "configured_limit": configured_limit,
        "count_over_limit": over_limit,
        "percent_over_limit": round(
            100 * over_limit / values.size,
            4,
        ),
    }

In [40]:
def analyze_sequence_lengths(
    backend_tokenizer: Tokenizer,
    train_df: pd.DataFrame,
    cfg: WordPieceConfig,
) -> dict[str, Any]:
    """Compute token-length statistics for separate and paired encodings.

    Batches the training questions to encode them with
    ``backend_tokenizer``, both individually (without special tokens) and
    as ``[CLS] question1 [SEP] question2 [SEP]`` pairs, then summarizes
    each length distribution with ``summarize_lengths``.

    Args:
        backend_tokenizer: Trained tokenizer backend used to encode the
            questions.
        train_df: Training split providing the ``question1`` and
            ``question2`` columns to analyze.
        cfg: Tokenizer configuration; supplies the length-analysis batch
            size and the configured length limits used for comparison.

    Returns:
        A dictionary with four length-statistics summaries: ``question1``,
        ``question2``, ``all_separate_questions`` (question1 and
        question2 combined), and ``paired_inputs``.
    """

    question1_lengths: list[int] = []
    question2_lengths: list[int] = []
    pair_lengths: list[int] = []

    batch_size = cfg.length_analysis_batch_size

    for start in range(
        0,
        len(train_df),
        batch_size,
    ):
        batch = train_df.iloc[
            start : start + batch_size
        ]

        question1_batch = (
            batch["question1"].tolist()
        )

        question2_batch = (
            batch["question2"].tolist()
        )

        question1_encodings = (
            backend_tokenizer.encode_batch(
                question1_batch,
                add_special_tokens=False,
            )
        )

        question2_encodings = (
            backend_tokenizer.encode_batch(
                question2_batch,
                add_special_tokens=False,
            )
        )

        pair_inputs = list(
            zip(
                question1_batch,
                question2_batch,
            )
        )

        pair_encodings = (
            backend_tokenizer.encode_batch(
                pair_inputs,
                add_special_tokens=True,
            )
        )

        question1_lengths.extend(
            len(encoding.ids)
            for encoding in question1_encodings
        )

        question2_lengths.extend(
            len(encoding.ids)
            for encoding in question2_encodings
        )

        pair_lengths.extend(
            len(encoding.ids)
            for encoding in pair_encodings
        )

    all_question_lengths = (
        question1_lengths
        + question2_lengths
    )

    statistics = {
        "question1": summarize_lengths(
            question1_lengths,
            cfg.max_sequence_length,
        ),
        "question2": summarize_lengths(
            question2_lengths,
            cfg.max_sequence_length,
        ),
        "all_separate_questions": summarize_lengths(
            all_question_lengths,
            cfg.max_sequence_length,
        ),
        "paired_inputs": summarize_lengths(
            pair_lengths,
            cfg.max_pair_length,
        ),
    }

    return statistics

In [41]:
length_statistics = analyze_sequence_lengths(
    backend_tokenizer=backend_tokenizer,
    train_df=train_df,
    cfg=cfg,
)

In [43]:
length_table = pd.DataFrame(
    {
        "separate_questions": (
            length_statistics[
                "all_separate_questions"
            ]
        ),
        "paired_inputs": (
            length_statistics[
                "paired_inputs"
            ]
        ),
    }
).T
length_table

,count,mean,p50,p90,p95,p99,max,configured_limit,count_over_limit,percent_over_limit
separate_questions,727716.0,13.168,11.0,22.0,27.0,38.0,282.0,64.0,661.0,0.0908
paired_inputs,363858.0,29.336,26.0,45.0,53.0,72.0,326.0,128.0,92.0,0.0253


## Section 11 - Training Metadata and Final Artifact Validation

`create_training_metadata` assembles everything worth recording about this run into one JSON-serializable dictionary: implementation details, the full tokenizer configuration, data provenance (source CSV, split sizes, corpus size), the trained vocabulary size and special-token ids, the list of saved files, and the sequence-length analysis from Section 10. It is written to `training_metadata.json` via `save_json`, giving downstream notebooks (and future readers) a single reference for exactly which configuration and data produced this vocabulary. The next two cells are closing checks: confirming the four required output files exist, and listing everything actually present in the tokenizer directory.

In [49]:
def create_training_metadata(
    cfg: WordPieceConfig,
    paths: WordPiecePaths,
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    corpus_question_count: int,
    tokenizer: PreTrainedTokenizerFast,
    length_statistics: dict[str, Any],
    saved_files: list[str],
) -> dict[str, Any]:
    """Assemble a JSON-serializable record of the tokenizer training run.

    Combines the tokenizer configuration, data provenance, vocabulary and
    special-token ids, the saved file list, and the sequence-length
    analysis into a single dictionary intended for ``save_json``.

    Args:
        cfg: Tokenizer configuration used for this training run.
        paths: Path configuration; supplies the source and output file
            paths recorded in the metadata.
        train_df: Training split used to train the tokenizer.
        valid_df: Validation split held out from tokenizer training.
        corpus_question_count: Number of question lines written to the
            training corpus.
        tokenizer: Trained fast tokenizer whose vocabulary size and
            special-token ids are recorded.
        length_statistics: Sequence-length statistics produced by
            ``analyze_sequence_lengths``.
        saved_files: File names written when the tokenizer was saved.

    Returns:
        A nested dictionary describing the implementation, configuration,
        data provenance, training results, and sequence-length analysis
        for this tokenizer.
    """

    return {
        "implementation": {
            "algorithm": "WordPiece",
            "backend": "Hugging Face Tokenizers",
            "wrapper": type(tokenizer).__name__,
        },

        "configuration": {
            **asdict(cfg),
            "special_tokens": cfg.special_tokens,
        },

        "data": {
            "source_csv": str(
                paths.train_csv_path
            ),
            "train_split": str(
                paths.train_split_path
            ),
            "validation_split": str(
                paths.valid_split_path
            ),
            "training_corpus": str(
                paths.corpus_path
            ),
            "train_rows": int(
                len(train_df)
            ),
            "validation_rows": int(
                len(valid_df)
            ),
            "corpus_questions": int(
                corpus_question_count
            ),
            "tokenizer_training_source": (
                "question1 and question2 "
                "from the training split only"
            ),
        },

        "training_results": {
            "actual_vocab_size": int(
                len(tokenizer)
            ),
            "special_token_ids": {
                "pad_token_id": (
                    tokenizer.pad_token_id
                ),
                "unk_token_id": (
                    tokenizer.unk_token_id
                ),
                "cls_token_id": (
                    tokenizer.cls_token_id
                ),
                "sep_token_id": (
                    tokenizer.sep_token_id
                ),
                "mask_token_id": (
                    tokenizer.mask_token_id
                ),
            },
            "saved_files": sorted(
                set(
                    saved_files
                    + [
                        paths.training_metadata_path.name
                    ]
                )
            ),
        },

        "sequence_length_analysis": (
            length_statistics
        ),
    }

In [51]:
training_metadata = create_training_metadata(
    cfg=cfg,
    paths=paths,
    train_df=train_df,
    valid_df=val_df,
    corpus_question_count=corpus_question_count,
    tokenizer=tokenizer,
    length_statistics=length_statistics,
    saved_files=saved_files,
)
save_json(
    training_metadata,
    paths.training_metadata_path,
)
training_metadata

>>> JSON saved to artifacts\tokenizers\wordpiece_uncased_30k\training_metadata.json...


{'implementation': {'algorithm': 'WordPiece',
  'backend': 'Hugging Face Tokenizers',
  'wrapper': 'PreTrainedTokenizerFast'},
 'configuration': {'vocab_size': 30000,
  'min_frequency': 2,
  'continuing_subword_prefix': '##',
  'max_input_chars_per_word': 100,
  'lowercase': True,
  'strip_accents': True,
  'clean_text': True,
  'handle_chinese_chars': True,
  'max_sequence_length': 64,
  'max_pair_length': 128,
  'pad_token': '[PAD]',
  'unk_token': '[UNK]',
  'cls_token': '[CLS]',
  'sep_token': '[SEP]',
  'mask_token': '[MASK]',
  'valid_size': 0.1,
  'seed': 28,
  'length_analysis_batch_size': 4096,
  'special_tokens': ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']},
 'data': {'source_csv': 'data\\raw\\train.csv',
  'train_split': 'data\\processed\\train_split.csv',
  'validation_split': 'data\\processed\\valid_split.csv',
  'training_corpus': 'data\\processed\\tokenizer_corpus.txt',
  'train_rows': 363858,
  'validation_rows': 40429,
  'corpus_questions': 727716,
  'tokenizer_trai

In [52]:
required_files = [
    paths.tokenizer_json_path,
    paths.tokenizer_config_path,
    paths.vocab_path,
    paths.training_metadata_path,
]

missing_files = [
    path.name
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing final files: {missing_files}"
    )

print(">>> Required tokenizer files exist.")

>>> Required tokenizer files exist.


In [53]:
final_files = sorted(
    path.name
    for path in paths.tokenizer_dir.iterdir()
    if path.is_file()
)

for filename in final_files:
    print(filename)

special_tokens_map.json
tokenizer.json
tokenizer_config.json
training_metadata.json
vocab.txt


### 11.1 End-to-End Sanity Check

A final smoke test loads the tokenizer fresh from disk exactly as a downstream training script would, encodes a brand-new question pair, and prints the resulting ids, token type ids, and decoded tokens. The closing message names the tokenizer's intended consumers explicitly — the LSTM-Attention, ESIM, and custom Transformer models — which is the reason two separate length budgets were configured back in Section 1.

In [55]:
final_tokenizer = (
    PreTrainedTokenizerFast.from_pretrained(
        str(paths.tokenizer_dir),
        local_files_only=True,
    )
)
final_test = final_tokenizer(
    "Why do people learn Python?",
    "What are the reasons for studying Python?",
    return_attention_mask=True,
    return_token_type_ids=True,
)
print(final_test)
print(
    final_tokenizer.convert_ids_to_tokens(
        final_test["input_ids"]
    )
)
print(
    ">>> Final tokenizer is ready for "
    "LSTM-Attention, ESIM and the custom Transformer."
)

{'input_ids': [2, 1499, 1462, 1617, 1681, 2989, 35, 3, 1455, 1475, 1449, 3464, 1484, 3296, 2989, 35, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
['[CLS]', 'why', 'do', 'people', 'learn', 'python', '?', '[SEP]', 'what', 'are', 'the', 'reasons', 'for', 'studying', 'python', '?', '[SEP]']
>>> Final tokenizer is ready for LSTM-Attention, ESIM and the custom Transformer.


## Conclusion

This notebook took the raw Quora Question Pairs CSV all the way to a trained, saved, and validated WordPiece tokenizer, along with a metadata record documenting exactly how it was produced.

| Stage | Goal | Key configuration / result |
|---|---|---|
| Configuration | Define paths and tokenizer hyperparameters | `vocab_size=30,000`, `max_sequence_length=64`, `max_pair_length=128` |
| Data loading and cleaning | Load QQP, drop invalid rows | 404,287 valid rows kept (3 removed) out of 404,290 |
| Train/validation split | Stratified 90/10 split | 363,858 train / 40,429 validation rows; duplicate rate ≈ 0.3692 in both |
| Corpus building | Write a train-only training corpus | 727,716 question lines |
| Tokenizer training | Train the WordPiece model | Actual vocabulary size: 30,000; all 5 special tokens present |
| Saving and validation | Save, then verify save/reload consistency | `tokenizer.json`, `tokenizer_config.json`, `vocab.txt`, `special_tokens_map.json` all confirmed on disk and behaviorally identical after reload |
| Length analysis | Check the configured length budgets | Separate questions: p99 ≈ 38, max 282, 0.0908% over the 64-token limit. Paired inputs: p99 ≈ 72, max 326, 0.0253% over the 128-token limit |

Based on the length-analysis table in Section 10, both configured budgets comfortably cover the vast majority of the corpus: fewer than one in a thousand examples exceed either limit, and the small tail that does will simply be truncated by the tokenizer's built-in truncation rather than causing an error. The two-budget design (64 tokens for single questions, 128 for pairs) reflects how each downstream architecture consumes the tokenizer's output, and the measured percentiles support both numbers as reasonable rather than arbitrary.

**Next steps.** The saved artifacts in `artifacts/tokenizers/wordpiece_uncased_30k/` are ready to be loaded — via `PreTrainedTokenizerFast.from_pretrained(...)` — by the LSTM-Attention, ESIM, and custom Transformer training notebooks that consume this tokenizer, using `training_metadata.json` as the reference for exactly what configuration and data produced it.

## **Resources & References**

**Official Documentation**
- [🤗 Tokenizers — Pipeline overview](https://huggingface.co/docs/tokenizers/pipeline)
- [🤗 Tokenizers — Trainers API (`WordPieceTrainer`)](https://huggingface.co/docs/tokenizers/main/en/api/trainers)
- [🤗 Transformers — Tokenizer main classes (`PreTrainedTokenizerFast`)](https://huggingface.co/docs/transformers/main_classes/tokenizer)
- [scikit-learn — `train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

**Key Papers / Research**
- Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2018). [*BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*](https://arxiv.org/abs/1810.04805).
- Wu, Y., et al. (2016). [*Google's Neural Machine Translation System: Bridging the Gap between Human and Machine Translation*](https://arxiv.org/abs/1609.08144) — introduces the WordPiece subword tokenization algorithm.

**Dataset**
- [Quora Question Pairs (Kaggle competition)](https://www.kaggle.com/c/quora-question-pairs)